# Nível 2 — Agente de Investigação de PLD

Neste nível será desenvolvido um agente capaz de investigar operações financeiras utilizando ferramentas determinísticas e um modelo de linguagem.

A abordagem separa as responsabilidades:
- Python executa consultas e cálculos sobre os dados;
- as ferramentas fornecem evidências objetivas;
- o LLM interpreta as evidências e produz o parecer final.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

pd.set_option("display.max_columns", None)

load_dotenv("../.env")

CAMINHO_DADOS = Path("../dados/dados_nivel_2.json")

print("Arquivo encontrado:", CAMINHO_DADOS.exists())

Arquivo encontrado: True


In [2]:
with open(CAMINHO_DADOS, "r", encoding="utf-8") as arquivo:
    dados_nivel_2 = json.load(arquivo)

taxa_cambio = dados_nivel_2["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados_nivel_2["operacoes"])

print(f"Taxa USD/BRL: {taxa_cambio}")
print(f"Quantidade inicial de registros: {len(df)}")
print(f"Quantidade de clientes: {df['cliente_id'].nunique()}")
print(f"Quantidade de IDs únicos: {df['id'].nunique()}")

display(df.head())

Taxa USD/BRL: 5.4
Quantidade inicial de registros: 322
Quantidade de clientes: 30
Quantidade de IDs únicos: 317


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-00133,CLI-014,2026-03-06,23640.97,BRL,pix,pagamento,Mirante Transportes ME,
1,OP-00103,CLI-011,2026-03-04,9447.52,BRL,cartao,saque,Quartzo Industria SA,
2,OP-00223,CLI-023,2026-04-13,2891.48,BRL,ted,transferencia_enviada,Gama Importacao SA,
3,OP-00265,CLI-028,2026-04-23,5636.46,BRL,pix,saque,Nauta Atacado ME,
4,OP-00099,CLI-010,2026-03-20,6641.24,BRL,ted,pagamento,Delta Trading LTDA,


In [3]:
print("Valores ausentes:")
display(df.isna().sum().to_frame("quantidade"))

print("\nIDs duplicados:")
duplicados = (
    df[df.duplicated(subset=["id"], keep=False)]
    .sort_values("id")
)

display(duplicados)

Valores ausentes:


,quantidade
id,0
cliente_id,0
data,7
valor,0
moeda,0
canal,0
tipo,0
contraparte,0
observacao,0



IDs duplicados:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
63,OP-00040,CLI-005,NaN,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
186,OP-00040,CLI-005,NaN,3025.05,BRL,especie,transferencia_enviada,Solar Importacao LTDA,data nao capturada pelo sistema
10,OP-00160,CLI-017,2026-03-19,3712.72,BRL,especie,saque,Lumen Industria ME,
143,OP-00160,CLI-017,2026-03-19,3712.72,BRL,especie,saque,Lumen Industria ME,
54,OP-00214,CLI-023,2026-03-20,1335.29,BRL,ted,saque,Mirante Consultoria SA,
253,OP-00214,CLI-023,2026-03-20,1335.29,BRL,ted,saque,Mirante Consultoria SA,
7,OP-00269,CLI-028,2026-05-23,6913.84,BRL,cartao,transferencia_enviada,Farol Distribuidora LTDA,
118,OP-00269,CLI-028,2026-05-23,6913.84,BRL,cartao,transferencia_enviada,Farol Distribuidora LTDA,
248,OP-00272,CLI-028,2026-03-27,6076.89,BRL,boleto,transferencia_enviada,Lumen Servicos SA,
266,OP-00272,CLI-028,2026-03-27,6076.89,BRL,boleto,transferencia_enviada,Lumen Servicos SA,


In [4]:
print("Moedas:")
display(df["moeda"].value_counts())

print("\nCanais:")
display(df["canal"].value_counts())

print("\nTipos:")
display(df["tipo"].value_counts())

Moedas:


moeda
BRL    315
USD      7
Name: count, dtype: int64


Canais:


canal
ted        82
especie    67
pix        61
cartao     58
boleto     54
Name: count, dtype: int64


Tipos:


tipo
deposito                  69
pagamento                 68
transferencia_enviada     68
transferencia_recebida    60
saque                     57
Name: count, dtype: int64

## Limpeza e normalização dos dados

Antes da investigação, os dados são normalizados para evitar que problemas de qualidade interfiram na análise.

São tratados:
- registros duplicados;
- datas ausentes;
- conversão de valores em USD para BRL;
- padronização dos tipos de dados.

Operações sem data conhecida são preservadas, mas não são utilizadas em análises que dependem de agrupamento temporal.

In [5]:
df_limpo = df.copy()

# Remove registros duplicados pelo ID da operação
df_limpo = df_limpo.drop_duplicates(subset=["id"], keep="first").copy()

# Converte a coluna de data
df_limpo["data"] = pd.to_datetime(df_limpo["data"], errors="coerce")

# Converte todos os valores para BRL
df_limpo["valor_brl"] = df_limpo.apply(
    lambda linha: (
        linha["valor"] * taxa_cambio
        if linha["moeda"] == "USD"
        else linha["valor"]
    ),
    axis=1
)

print(f"Registros antes da limpeza: {len(df)}")
print(f"Registros depois da limpeza: {len(df_limpo)}")
print(f"Duplicados restantes: {df_limpo['id'].duplicated().sum()}")
print(f"Datas ausentes preservadas: {df_limpo['data'].isna().sum()}")

display(df_limpo.head())

Registros antes da limpeza: 322
Registros depois da limpeza: 317
Duplicados restantes: 0
Datas ausentes preservadas: 6


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl
0,OP-00133,CLI-014,2026-03-06,23640.97,BRL,pix,pagamento,Mirante Transportes ME,,23640.97
1,OP-00103,CLI-011,2026-03-04,9447.52,BRL,cartao,saque,Quartzo Industria SA,,9447.52
2,OP-00223,CLI-023,2026-04-13,2891.48,BRL,ted,transferencia_enviada,Gama Importacao SA,,2891.48
3,OP-00265,CLI-028,2026-04-23,5636.46,BRL,pix,saque,Nauta Atacado ME,,5636.46
4,OP-00099,CLI-010,2026-03-20,6641.24,BRL,ted,pagamento,Delta Trading LTDA,,6641.24


## Regras determinísticas em escala

As mesmas regras do Nível 1 são reaplicadas sobre a base maior após a limpeza dos dados.

- Regra 1: fracionamento de operações;
- Regra 2: valor atípico em relação à mediana do cliente.

In [6]:
df_limpo["flag_fracionamento"] = False

df_com_data = df_limpo.dropna(subset=["data"]).copy()

resumo_dia = (
    df_com_data
    .groupby(["cliente_id", "data"])
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_dia_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max"),
    )
    .reset_index()
)

casos_fracionamento = resumo_dia[
    (resumo_dia["quantidade_operacoes"] >= 3)
    & (resumo_dia["soma_dia_brl"] > 50000)
    & (resumo_dia["maior_operacao_brl"] < 20000)
].copy()

for _, caso in casos_fracionamento.iterrows():
    mascara = (
        (df_limpo["cliente_id"] == caso["cliente_id"])
        & (df_limpo["data"] == caso["data"])
    )
    df_limpo.loc[mascara, "flag_fracionamento"] = True

print("Casos de fracionamento encontrados:")
display(casos_fracionamento)

Casos de fracionamento encontrados:


,cliente_id,data,quantidade_operacoes,soma_dia_brl,maior_operacao_brl
15,CLI-002,2026-05-01,4,64723.09,17998.60
28,CLI-003,2026-05-02,4,50846.72,18631.47
148,CLI-017,2026-03-08,4,64673.88,18761.22
276,CLI-029,2026-05-26,4,71297.68,19418.96


In [7]:
estatisticas_cliente = (
    df_limpo
    .groupby("cliente_id")
    .agg(
        quantidade_operacoes=("id", "count"),
        mediana_valor_brl=("valor_brl", "median"),
    )
    .reset_index()
)

df_limpo = df_limpo.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left",
)

df_limpo["flag_valor_atipico"] = (
    (df_limpo["quantidade_operacoes"] >= 4)
    & (df_limpo["valor_brl"] > 5 * df_limpo["mediana_valor_brl"])
)

print("Operações com valor atípico:")
display(
    df_limpo[df_limpo["flag_valor_atipico"]][
        [
            "id",
            "cliente_id",
            "valor_brl",
            "mediana_valor_brl",
            "quantidade_operacoes",
        ]
    ]
)

Operações com valor atípico:


,id,cliente_id,valor_brl,mediana_valor_brl,quantidade_operacoes
0,OP-00133,CLI-014,23640.970,2308.410,11
18,OP-00197,CLI-021,15785.390,2832.545,10
19,OP-00129,CLI-014,13660.650,2308.410,11
24,OP-00253,CLI-026,21261.010,2032.930,12
52,OP-00219,CLI-023,41768.170,3241.160,12
82,OP-00008,CLI-001,25110.150,1609.155,10
110,OP-00049,CLI-005,11988.170,2144.180,11
117,OP-00310,CLI-013,28487.760,3146.225,10
121,OP-00312,CLI-022,30894.858,1909.890,11
126,OP-00316,CLI-024,68247.144,2740.780,11


In [8]:
resumo_clientes = (
    df_limpo
    .groupby("cliente_id")
    .agg(
        sinalizacoes_fracionamento=("flag_fracionamento", "sum"),
        sinalizacoes_valor_atipico=("flag_valor_atipico", "sum"),
        volume_total_brl=("valor_brl", "sum"),
    )
    .reset_index()
)

resumo_clientes["total_sinalizacoes"] = (
    resumo_clientes["sinalizacoes_fracionamento"]
    + resumo_clientes["sinalizacoes_valor_atipico"]
)

top_10_clientes = (
    resumo_clientes
    .sort_values(
        ["total_sinalizacoes", "volume_total_brl"],
        ascending=[False, False],
    )
    .head(10)
    .reset_index(drop=True)
)

display(top_10_clientes)

,cliente_id,sinalizacoes_fracionamento,sinalizacoes_valor_atipico,volume_total_brl,total_sinalizacoes
0,CLI-029,4,0,191385.766,4
1,CLI-017,4,0,121391.370,4
2,CLI-002,4,0,107965.380,4
3,CLI-003,4,0,102241.090,4
4,CLI-014,0,3,80629.990,3
5,CLI-023,0,2,148535.016,2
6,CLI-028,0,2,88750.800,2
7,CLI-013,0,2,81730.990,2
8,CLI-005,0,2,64742.660,2
9,CLI-026,0,2,54729.280,2


## Ferramentas do agente

Antes da construção do agente, as ferramentas de investigação são testadas individualmente para verificar seu funcionamento.

In [9]:
from tools import (
    buscar_operacoes_cliente,
    calcular_estatisticas_cliente,
    buscar_operacoes_sinalizadas,
    buscar_contrapartes,
)

In [10]:
cliente_teste = "CLI-029"

teste_estatisticas = calcular_estatisticas_cliente(
    df_limpo,
    cliente_teste
)

teste_sinalizadas = buscar_operacoes_sinalizadas(
    df_limpo,
    cliente_teste
)

teste_contrapartes = buscar_contrapartes(
    df_limpo,
    cliente_teste
)

print("ESTATÍSTICAS:")
print(teste_estatisticas)

print("\nOPERAÇÕES SINALIZADAS:")
display(pd.DataFrame(teste_sinalizadas))

print("\nCONTRAPARTES:")
display(pd.DataFrame(teste_contrapartes))

ESTATÍSTICAS:
{'cliente_id': 'CLI-029', 'quantidade_operacoes': 16, 'volume_total_brl': 191385.766, 'media_valor_brl': 11961.610375, 'mediana_valor_brl': 10337.485, 'maior_operacao_brl': 48045.636000000006, 'menor_operacao_brl': 842.71}

OPERAÇÕES SINALIZADAS:


,id,data,valor_brl,canal,tipo,contraparte,flag_fracionamento,flag_valor_atipico
0,OP-00301,2026-05-26,19418.96,ted,saque,Orion Trading SA,True,False
1,OP-00300,2026-05-26,19138.59,ted,deposito,Solar Importacao LTDA,True,False
2,OP-00299,2026-05-26,14326.29,ted,transferencia_recebida,Rubi Importacao ME,True,False
3,OP-00302,2026-05-26,18413.84,pix,deposito,Cristal Atacado ME,True,False



CONTRAPARTES:


,contraparte,quantidade_operacoes,volume_total_brl
0,Boreal Varejo ME,1,48045.636
1,Rubi Importacao ME,2,26835.450
2,Orion Trading SA,1,19418.960
3,Solar Importacao LTDA,1,19138.590
4,Cristal Atacado ME,1,18413.840
5,Tijuca Transportes ME,1,16474.170
6,Pampa Varejo LTDA,1,12407.610
7,Lumen Servicos LTDA,1,8267.360
8,Estrela Atacado ME,1,6305.910
9,Gama Transportes ME,1,4707.590


## Teste do agente de investigação

O agente recebe apenas o identificador do cliente e decide autonomamente quais ferramentas utilizar durante a investigação.

In [14]:
import importlib
import agente

importlib.reload(agente)

from agente import investigar_cliente

In [15]:
resultado_agente = investigar_cliente(
    df_limpo,
    "CLI-029"
)

print("Cliente:", resultado_agente["cliente_id"])

print("\nFerramentas escolhidas pelo agente:")
print(resultado_agente["ferramentas_usadas"])

print("\nParecer:")
print(resultado_agente["parecer"])

Cliente: CLI-029

Ferramentas escolhidas pelo agente:
['buscar_operacoes_sinalizadas', 'buscar_operacoes_cliente', 'calcular_estatisticas_cliente', 'buscar_contrapartes']

Parecer:
**Parecer de Investigação – Cliente CLI‑029**

| Item | Informação |
|------|------------|
| **cliente_id** | CLI‑029 |
| **nível_risco** | **Alto** |
| **tipologia_suspeita** | Estruturação (fracionamento) e movimentação de valores elevados em curto prazo, com possível tentativa de evitar limites de reporte. |
| **principais_evidências** | 1. **Quatro operações sinalizadas** (OP‑00301, OP‑00300, OP‑00299, OP‑00302) realizadas no mesmo dia (26/05/2026) com valores entre R$ 18.3 k e R$ 19.4 k, todas marcadas com `flag_fracionamento = true`. <br>2. **Depósito de R$ 48.045,636** em 13/04/2026 (OP‑00311) com observação “remessa internacional”. <br>3. **Volume total do cliente**: R$ 191 385,77 em 16 operações, média de R$ 11 962,61 por operação. <br>4. **Diversas contrapartes** (Orion Trading SA, Solar Importação

In [16]:
clientes_teste = ["CLI-029", "CLI-014", "CLI-023"]

resultados_teste_agente = []

for cliente_id in clientes_teste:
    resultado = investigar_cliente(df_limpo, cliente_id)

    resultados_teste_agente.append({
        "cliente_id": cliente_id,
        "ferramentas_usadas": resultado["ferramentas_usadas"],
        "parecer": resultado["parecer"]
    })

for resultado in resultados_teste_agente:
    print("=" * 80)
    print("CLIENTE:", resultado["cliente_id"])
    print("FERRAMENTAS:", resultado["ferramentas_usadas"])
    print("\nPARECER:")
    print(resultado["parecer"])
    print()

CLIENTE: CLI-029
FERRAMENTAS: ['buscar_operacoes_sinalizadas', 'buscar_operacoes_cliente', 'calcular_estatisticas_cliente', 'buscar_contrapartes']

PARECER:
**Parecer de Investigação – Cliente CLI‑029**

| Item | Informação |
|------|------------|
| **cliente_id** | CLI‑029 |
| **nivel_risco** | **Alto** |
| **tipologia_suspeita** | Estruturação de operações (structuring) e movimentação de fundos de origem potencialmente ilícita |
| **principais_evidências** | 1. **Quatro operações sinalizadas** (OP‑00301, OP‑00300, OP‑00299, OP‑00302) realizadas no mesmo dia (26‑05‑2026) com valores superiores a R$ 14 000 cada. <br>2. Todas as operações sinalizadas apresentam **flag_fracionamento = true**, indicando que foram divididas em partes menores, típico de tentativa de evitar limites de reporte. <br>3. Operações envolvem **canais variados** (TED, PIX) e **contrapartes** distintas, sugerindo tentativa de dispersão de rastreamento. <br>4. O cliente possui um depósito de R$ 48 045,63 (OP‑00311) c